# StormCast Inference Workshop

**Part 2 of 2** | [Part 1: Training Notebook](./stormcast_training_notebook.ipynb)

---

This notebook demonstrates how to run inference with **StormCastMini**, the model you trained in Part 1. We'll load the exported model and run forecasts using NVIDIA's Earth2Studio framework.

## Learning Objectives

By the end of this notebook, you will understand:

1. **StormCastMini class structure**: How our model integrates with Earth2Studio
2. **Model loading**: How to load exported checkpoints and metadata
3. **Earth2Studio workflows**: The standardized inference pipeline
4. **Visualization**: How to interpret forecast outputs

## Prerequisites

- **Completed Part 1** (Training Notebook) with an exported model package
- Understanding of EDM diffusion (covered in Part 1)
- Familiarity with HRRR/ERA5 data (covered in Part 1)

---

## 1. The StormCastMini Class

In Part 1 (Training), we trained a diffusion model. Now we need to wrap it in a class that Earth2Studio understands. That's what `StormCastMini` does.

### Why a Custom Class?

Earth2Studio's official `StormCast` class expects:
- A **regression model** (required) + **diffusion model**
- The full 99-channel HRRR variable set
- Fixed CONUS grid limits

Our workshop model is simpler:
- **Diffusion-only** (no regression model)
- Custom variables (u10m, v10m, t2m)
- Small regional grid (Bay Area)

### Class Hierarchy

```
torch.nn.Module
    └── StormCast (Earth2Studio)     ← Handles coordinates, data fetching, workflows
            └── StormCastMini        ← Our customizations
```

### What StormCastMini Overrides

| Method | Purpose | Why Override? |
|--------|---------|---------------|
| `__init__` | Initialize model | Make regression model optional |
| `_forward` | Single prediction step | Support diffusion-only mode |
| `__call__` | Earth2Studio interface | Handle optional conditioning |
| `load_model` | Load from package | Custom package format |

### What We Inherit (No Changes Needed)

- `input_coords()` / `output_coords()` → Coordinate system handling
- `create_iterator()` → Autoregressive rollout
- Batch processing decorators → Multi-sample inference

This is the power of subclassing: we get all the Earth2Studio integration for free!

---

## 2. Key Code Walkthrough

Let's look at the most important parts of `StormCastMini`. The full code is in `stormcast_mini.py`.

### The Forward Pass (`_forward` method)

This is where the magic happens:

```python
def _forward(self, x, conditioning=None):
    # 1. Normalize input state
    x_norm = (x - self.means) / self.stds
    
    # 2. Build diffusion condition
    #    Concatenate: [state, conditioning, invariants]
    diff_condition = torch.cat((x_norm, conditioning, invariants), dim=1)
    
    # 3. Sample random noise
    latents = torch.randn_like(x_norm)
    
    # 4. Run EDM diffusion sampler
    #    This iteratively denoises the latents
    edm_out = deterministic_sampler(
        self.diffusion_model,
        latents=latents,
        img_lr=diff_condition,  # conditioning
        **self.sampler_args,
    )
    
    # 5. Denormalize and return
    return edm_out * self.stds + self.means
```

### The Conditioning Flow

```
┌──────────────┐     ┌─────────────────┐     ┌──────────────┐
│ HRRR State t │     │ ERA5 Global t+1 │     │  Invariants  │
│   (3 ch)     │     │     (26 ch)     │     │   (3 ch)     │
└──────┬───────┘     └────────┬────────┘     └──────┬───────┘
       │                      │                     │
       └──────────────────────┼─────────────────────┘
                              ▼
                     ┌────────────────┐
                     │  Concatenate   │
                     │   (32 ch)      │
                     └────────┬───────┘
                              ▼
                     ┌────────────────┐
                     │   Diffusion    │
                     │   U-Net        │
                     └────────┬───────┘
                              ▼
                     ┌─────────────────┐
                     │  Prediction t+1 │
                     │     (3 ch)      │
                     └─────────────────┘
```

---

## 3. Setup and Imports

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, "/dli/workshop")

# Set cache directory
import os
os.environ["EARTH2STUDIO_CACHE"] = "/output/"

# Set verbosity level, use "DEBUG" for more verbose output
os.environ["LOGURU_LEVEL"] = "INFO"

from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

# Earth2Studio components
from earth2studio.data import ARCO, HRRR
from earth2studio.io import ZarrBackend
import earth2studio.run as run

# Our workshop model
from scripts.stormcast_mini import StormCastMini

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

---

## 4. Load the Model

We load our trained StormCastMini model from the package exported in Part 1.

### Package Contents (from Training Notebook)

| File | Contents |
|------|----------|
| `model.yaml` | Configuration (variables, sampler args) |
| `EDMPrecond.*.mdlus` | Diffusion model weights |
| `metadata.zarr.zip` | Normalization stats, variable names, invariants |
| `StormCastUNet.*.mdlus` | Regression model (optional, not in our workshop) |

In [ ]:
# Path to your exported model package
PACKAGE_PATH = Path("/output/model_package")

# Verify package exists
if PACKAGE_PATH.exists():
    print("Model package contents:")
    for f in sorted(PACKAGE_PATH.iterdir()):
        print(f"  - {f.name}")
else:
    print(f"⚠️  Package not found at {PACKAGE_PATH}")
    print("Run the training notebook first to export a model!")

In [ ]:
# Create conditioning data source
# ARCO provides ERA5 reanalysis data for global context
conditioning_data_source = ARCO()

# Load the model
model = StormCastMini.load_model(
    PACKAGE_PATH,
    conditioning_data_source=conditioning_data_source,
)

# Move to GPU and set to eval mode
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device).eval()

print(f"\n{'='*50}")
print(f"Model loaded successfully!")
print(f"{'='*50}")
print(f"Device: {device}")
print(f"Variables: {list(model.variables)}")
print(f"Has regression model: {model.regression_model is not None}")
print(f"Grid size: {len(model.hrrr_y)} x {len(model.hrrr_x)}")
print(f"Lat range: [{model.lat.min():.2f}, {model.lat.max():.2f}]")
print(f"Lon range: [{model.lon.min():.2f}, {model.lon.max():.2f}]")

---

## 5. Earth2Studio Workflow

Earth2Studio provides a standardized interface for running weather AI models.

### The Workflow Pattern

```python
# 1. Data source for initial conditions
data = HRRR()  # Fetches real HRRR analysis data

# 2. Model with conditioning source
model = StormCastMini.load_model(..., conditioning_data_source=ARCO())

# 3. Output storage
io = ZarrBackend()  # Stores results in Zarr format

# 4. Run forecast
io = run.deterministic([date], nsteps, model, data, io)
```

### Data Sources

| Source | Description | Use Case |
|--------|-------------|----------|
| `HRRR()` | NOAA's 3km CONUS model | Initial state |
| `ARCO()` | ERA5 reanalysis archive | Global conditioning |
| `GFS_FX()` | GFS forecast data | Operational conditioning |

In [ ]:
# Create data source for initial state
data = HRRR()

# Create output storage
io = ZarrBackend()

# Forecast parameters
nsteps = 4  # 4 hours ahead

# Pick a date (must be available in HRRR archive, typically 2014-present)
# Using a date ~5.5 years ago to ensure data availability
forecast_date = "2022-05-21T18:00:00"

print(f"Forecast Configuration")
print(f"{'='*50}")
print(f"Date: {forecast_date}")
print(f"Forecast length: {nsteps} hours")
print(f"Initial state source: {type(data).__name__}")
print(f"Conditioning source: {type(model.conditioning_data_source).__name__}")

---

## 6. Run the Forecast

Now we run the model using Earth2Studio's deterministic workflow.

Behind the scenes, this:
1. Fetches HRRR initial conditions for our region
2. At each time step, fetches ERA5 conditioning data
3. Runs the diffusion model to generate the next hour's prediction
4. Stores results in the Zarr backend

In [ ]:
# Run the forecast!
io = run.deterministic([forecast_date], nsteps, model, data, io)

print("\nForecast complete!")
print("\nOutput structure:")
print(io.root.tree())

---

## 7. Visualize Results

Let's examine the forecast output and compare different lead times.

In [ ]:
# Extract data from the Zarr backend
variables = list(model.variables)
lead_times = io["lead_time"][:]

print(f"Variables: {variables}")
print(f"Lead times: {lead_times}")

In [ ]:
from scripts.plots import plot_forecast_evolution

# Plot each variable
for var in variables:
    plot_forecast_evolution(io, var)
    plt.show()

---

## 8. Interpreting Forecasts

### Quality Checklist

When evaluating your model's forecasts, look for:

| Aspect | Good Signs | Warning Signs |
|--------|------------|---------------|
| **Temporal coherence** | Smooth evolution | Jumpy/flickering patterns |
| **Spatial structure** | Realistic gradients | Blocky artifacts, checkerboard |
| **Physical consistency** | Wind/temp correlation | Unphysical combinations |
| **Value ranges** | Within climatology | Extreme outliers |

### Workshop Model Limitations

Remember from Part 1 (Training Notebook), our StormCastMini is intentionally simplified:

| Our Model | Full StormCast |
|-----------|----------------|
| 3 variables | 99 HRRR channels |
| 64×64 grid | 512×640 grid |
| Hours of training | Days of training |

If you see artifacts, it's likely due to limited training rather than a fundamental model issue.

---

## 9. Tasks

---

**Task 1: Run a real-time forecast**

Run a real-time forecast initialized with the most recent HRRR and GFS initial conditions over San Jose.

Steps:
1. Set the date to the last available IC (00, 06, 12 or 18 UTC)
2. Initialize StormCastMini with GFS_FX conditioning data source
3. Run and visualize the forecast

<details>
<summary><b>Solution</b></summary>

```python
from earth2studio.data import HRRR, GFS_FX
from datetime import datetime, timedelta
import numpy as np

# Forecast setup
forecast_dt = datetime.utcnow()
forecast_dt = forecast_dt.replace(hour=((forecast_dt.hour) // 6) * 6, minute=0, second=0, microsecond=0)
forecast_date = forecast_dt.strftime("%Y-%m-%dT%H")
print(f"Forecast date: {forecast_date}")

# Set GFS_FX as conditioning data source
conditioning_data_source = GFS_FX()
# Load the model
model = StormCastMini.load_model(
    PACKAGE_PATH,
    conditioning_data_source=conditioning_data_source,
)

# Set number of steps
nsteps = 6

# Initialize I/O backend
io_forecast = ZarrBackend()

# Run forecast
io_forecast = run.deterministic([forecast_date], nsteps, model, data, io_forecast)

# Visualize the forecast
variables = list(model.variables)

# Plot each variable
for var in variables:
    plot_forecast_evolution(io_forecast, var)
    plt.show()
```
</details>

In [ ]:
# Task 1: Your code here

**Task 2: Measure Model Performance vs Ground Truth**

Compare your model's predictions against the actual HRRR analysis to compute error metrics. Additionally, fetch HRRR forecasts and compare StormCastMini vs them. 
Optional: run a persistence model too. 

Steps:
1. Run a forecast for a specific date
2. Fetch the HRRR ground truth for the same valid times
3. Compute RMSE for each variable

</details>

<details>
<summary><b>Solution</b></summary>

```python
from earth2studio.data import HRRR_FX, HRRR, GFS_FX
from datetime import datetime, timedelta
import numpy as np

# Forecast setup
forecast_date = "2025-05-21T18:00:00"
print(f"Forecast date: {forecast_date}")

# Set GFS_FX as conditioning data source
conditioning_data_source = GFS_FX()
# Load the model
model = StormCastMini.load_model(
    PACKAGE_PATH,
    conditioning_data_source=conditioning_data_source,
)

# Run forecast
io_forecast = ZarrBackend()
io_forecast = run.deterministic([forecast_date], nsteps, model, data, io_forecast)

# Fetch HRRR analysis and forecast
hrrr_fx = HRRR_FX()
hrrr = HRRR()
lead_times = io_forecast["lead_time"][:]
print(lead_times)

# Get model's grid limits (from load_model defaults or the loaded model)
# These are the HRRR grid indices for the regional subset
hrrr_lat_lim = (549, 613)  # y indices
hrrr_lon_lim = (157, 221)  # x indices

init_time = np.datetime64(forecast_date)

# Compute errors for each variable
print("\n--- Error Analysis ---")
lead_times = np.array([np.timedelta64(lt, 'h') for lt in lead_times])
hrrr_pred = hrrr_fx(np.array([init_time]), lead_times, model.variables)
hrrr_truth = hrrr(np.array([init_time + lt for lt in lead_times]), model.variables)

# Get initial state for persistence baseline (forecast = initial condition for all lead times)
hrrr_initial = hrrr(np.array([init_time]), model.variables)

for i, var in enumerate(model.variables):
    model_var_pred = io_forecast[var][0]
    hrrr_var_pred = hrrr_pred[0, :, i, hrrr_lat_lim[0]:hrrr_lat_lim[1], hrrr_lon_lim[0]:hrrr_lon_lim[1]]
    hrrr_var_truth = hrrr_truth[:, i, hrrr_lat_lim[0]:hrrr_lat_lim[1], hrrr_lon_lim[0]:hrrr_lon_lim[1]]
    
    # Persistence: use initial state as forecast for all lead times
    hrrr_var_initial = hrrr_initial[0, i, hrrr_lat_lim[0]:hrrr_lat_lim[1], hrrr_lon_lim[0]:hrrr_lon_lim[1]]
    # Repeat initial state for each lead time
    persistence_pred = np.broadcast_to(hrrr_var_initial.values, hrrr_var_truth.shape)
    
    plt.figure()
    plt.title(var)
    plt.plot(np.sqrt((model_var_pred - hrrr_var_truth.values)**2).mean(axis=(1,2)), label="StormCastMini")
    plt.plot(np.sqrt((hrrr_var_pred.values - hrrr_var_truth.values)**2).mean(axis=(1,2)), label="HRRR")
    plt.plot(np.sqrt((persistence_pred - hrrr_var_truth.values)**2).mean(axis=(1,2)), label="Persistence")
    plt.legend()
    plt.ylabel("RMSE")
    plt.xlabel("Lead Time [h]")
    plt.show()
```
</details>

In [ ]:
# Task 2: Your code here

---

**Task 3: Diffusion Steps vs Quality Tradeoff**

The diffusion sampler uses `num_steps` to control the number of denoising steps. More steps = better quality but slower inference.

Experiment with different values and measure:
- Time per forecast
- Visual quality differences

<details>
<summary><b>Solution</b></summary>

```python
import time
from earth2studio.data import GFS_FX

# Forecast setup
forecast_date = "2025-05-21T18:00:00"
print(f"Forecast date: {forecast_date}")

# Set GFS_FX as conditioning data source
conditioning_data_source = GFS_FX()

# Load the model
model = StormCastMini.load_model(
    PACKAGE_PATH,
    conditioning_data_source=conditioning_data_source,
)

step_configs = [5, 25, 50, 100]
results = {}

for num_steps in step_configs:
    # Update sampler settings
    model.sampler_args["num_steps"] = num_steps
    
    # Time the forecast
    io_test = ZarrBackend()
    start = time.time()
    io_test = run.deterministic([forecast_date], nsteps, model, data, io_test)
    elapsed = time.time() - start
    
    results[num_steps] = {
        'time': elapsed,
        'output': io_test['t2m'][0, -1]  # Last timestep of t2m
    }
    print(f"num_steps={num_steps}: {elapsed:.2f}s")

# Visualize differences
fig, axes = plt.subplots(1, len(step_configs), figsize=(16, 4))
for ax, steps in zip(axes, step_configs):
    ax.imshow(results[steps]['output'], cmap='RdBu_r')
    ax.set_title(f'steps={steps}\n{results[steps]["time"]:.1f}s')
    ax.axis('off')
plt.suptitle('Diffusion Steps Comparison')
plt.tight_layout()
```
</details>

In [ ]:
# Task 3: Your code here

---

**Challenge Task: Use SFNO Forecasts as Conditioning**

Instead of using ERA5 or GFS for conditioning, use predictions from NVIDIA's SFNO (Spherical Fourier Neural Operator) global model.

This creates a cascaded forecast: SFNO (global) → StormCast (regional)

<details>
<summary><b>Hint</b></summary>

You'll need to:
1. Load SFNO from Earth2Studio
2. Run a global forecast
3. Temporally interpolate the global forecast to get 1-hourly output
4. Use the SFNO output as conditioning for StormCast with the InferenceOutputSource data source


```python
from earth2studio.models.px import SFNO, InterpModAFNO
from earth2studio.data import GFS, InferenceOutputSource

# Load SFNO
sfno_package = SFNO.load_default_package()
sfno_model = SFNO.load_model(sfno_package)

# The challenge is creating a data source wrapper around SFNO output
# that StormCast can use for conditioning...
```

</details>

<details>
<summary><b>Solution</b></summary>

```python
from earth2studio.models.px import SFNO, InterpModAFNO
from earth2studio.data import GFS, InferenceOutputSource  # For SFNO initial conditions

nsteps = 6
forecast_date = "2022-05-21T18:00:00"

# Step 1: Run SFNO global forecast
sfno_package = SFNO.load_default_package()
px_model = SFNO.load_model(sfno_package)

# Load the interpolation model
interp_package = InterpModAFNO.load_default_package()
interp_model = InterpModAFNO.load_model(interp_package, px_model=px_model)

sfno_data = GFS()  # SFNO uses GFS for initialization
sfno_io = ZarrBackend("/output/sfno_forecast.zarr", backend_kwargs={"overwrite": True})
sfno_io = run.deterministic(
    [forecast_date],
    nsteps,
    interp_model,
    sfno_data,
    sfno_io,
    output_coords={"variable": model.conditioning_variables},
)

# Step 3: Use SFNO output for StormCast conditioning
sfno_source = InferenceOutputSource("/output/sfno_forecast.zarr", {"variable": model.conditioning_variables})
model.conditioning_data_source = sfno_source

# Run cascaded forecast
io_cascaded = ZarrBackend()
io_cascaded = run.deterministic([forecast_date], nsteps, model, data, io_cascaded)

# Visualize the forecast
variables = list(model.variables)

# Plot each variable
for var in variables:
    plot_forecast_evolution(io_cascaded, var)
    plt.show()
```

</details>

In [ ]:
# Challenge: Your code here

---

## Summary

### Workshop Complete! 🎉

You've completed both parts of the StormCast workshop:

| Part | What You Learned |
|------|------------------|
| **Part 1: Training** | Data preparation, EDM diffusion theory, model training |
| **Part 2: Inference** | Model loading, Earth2Studio workflows, forecast visualization |

### Key Takeaways

1. **StormCastMini extends StormCast** - Minimal subclassing for custom configurations
2. **Earth2Studio handles complexity** - Data fetching, coordinate systems, workflows
3. **EDM diffusion generates realistic weather** - Learned denoising → conditional generation
4. **Model quality depends on training** - More data, more steps, better forecasts

### Continue Learning

- **Full StormCast**: [nvidia/stormcast-v1-era5-hrrr](https://huggingface.co/nvidia/stormcast-v1-era5-hrrr) on Hugging Face
- **Earth2Studio docs**: https://nvidia.github.io/earth2studio/
- **StormCast paper**: https://arxiv.org/abs/2408.10958
- **PhysicsNeMo**: https://github.com/NVIDIA/physicsnemo